[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MahkameSalimi/ISPRS-Tutorial/blob/main/intro_quantum_and_lstm/intro_to_lstm_pytorch.ipynb)

# Introduction to LSTM

Many real-world problems involve sequential data, where the order of observations carries meaning. Two broad examples of sequential data inlcude time series dataset and text. For instance, temperature and stock pricing over time are instance of time-series datasets, while natural language is an example of text sequentil data. In contrast to sequential data, tabular data are instance of static data, where the ordering is not a relevant factor.

To handle sequential data, machine learning algorithms should have some sort of notion of memory to be able to process such information and connect to earlier data points to understand relations better for better understaind. As such, recurrent neural networks were developed.

These models are suitable for short-range dependencies; however, they come with their own limitaitons, namely learning long-term dependencies, as it runs into computational problems such as vanishing graidents.

Long short-term memories were deveoloped to address the limitations of RNNs, so they can maintain memory over longer range while remaining computaionally feasible.


This notebook is a short hands-on introduction to **Long Short-Term Memory networks (LSTMs)**.

By the end, you should understand:

- what sequence data is
- why ordinary feedforward neural networks are not enough for sequences
- what a recurrent neural network (RNN) does
- why vanilla RNNs can struggle with long-term memory
- how an LSTM uses gates to control memory
- how to build and train a small LSTM model in PyTorch
- how this connects to QLSTM later

## 1. What is sequence data?

Sequence data means the order of the data points matters.

Examples:

- words in a sentence
- stock prices over time
- temperature over days
- river level measurements over time
- neural spike trains
- sensor readings

For example, in a time series:

$$
x_1, x_2, x_3, \dots, x_T
$$

Each $x_t$ is the input at time step $t$.

The goal may be to predict:

$$
x_{T+1}
$$

or to classify the whole sequence.

## 2. Why not just use a normal neural network?

A normal feedforward neural network treats inputs mostly as fixed-size vectors.

For example:

$$
y = f(x)
$$

But for sequence data, we often need the model to remember previous information.

For example, if we observe:

$$
x_1, x_2, x_3, \dots, x_t
$$

then the prediction at time $t$ may depend on earlier values, not only the current value.

This is why we use recurrent models.

## 3. Vanilla RNN idea

A recurrent neural network keeps a hidden state.

At each time step:

$$
h_t = \tanh(W_x x_t + W_h h_{t-1} + b)
$$

where:

- $x_t$ is the current input
- $h_{t-1}$ is the previous hidden state
- $h_t$ is the updated hidden state
- $W_x$, $W_h$, and $b$ are trainable parameters

The hidden state acts like memory.

But vanilla RNNs often struggle to remember information from many time steps ago. This is related to the **vanishing gradient problem**.


## 4. LSTM intuition
![Unrolled LSTM over time](RNN.png)

**DOI:** [10.1162/neco.1997.9.8.1735](https://doi.org/10.1162/neco.1997.9.8.1735)

**License:** © 1997 Massachusetts Institute of Technology. Published in *Neural Computation* by MIT Press. Reproduced for educational use; reuse beyond this tutorial requires permission from the publisher.

## 5. LSTM equations
![Unrolled LSTM over time](LSTM.png)

**DOI:** [10.1162/neco.1997.9.8.1735](https://doi.org/10.1162/neco.1997.9.8.1735)

**License:** © 1997 Massachusetts Institute of Technology. Published in *Neural Computation* by MIT Press. Reproduced for educational use; reuse beyond this tutorial requires permission from the publisher.

## 6. Big picture of one LSTM step

At time step $t$:

$$
x_t, h_{t-1}, c_{t-1} \longrightarrow h_t, c_t
$$

So the LSTM cell is a function:

$$
(h_t, c_t) = \mathrm{LSTMCell}(x_t, h_{t-1}, c_{t-1})
$$

The same LSTM cell is reused at every time step.

That means the same trainable weights are shared through time.

## 7. Install/import libraries

This notebook uses PyTorch.

If PyTorch is not installed, uncomment the install command below.

In [ ]:
# !pip install torch matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Make results reproducible
np.random.seed(42)
torch.manual_seed(42)

## 8. Create a simple time-series dataset

We will generate a sine wave and train an LSTM to predict the next value.

The data looks like:

$$
x_t = \sin(t)
$$

Given a window of previous values, the model predicts the next value.

In [ ]:
# Generate a sine wave
T = 500
time = np.linspace(0, 20 * np.pi, T)
data = np.sin(time)

plt.figure(figsize=(10, 4))
plt.plot(time, data)
plt.title("Synthetic sine-wave time series")
plt.xlabel("time")
plt.ylabel("x(t)")
plt.show()

## 9. Convert the time series into supervised learning samples

We use a sliding window.

For example, if the sequence length is 20:

$$
[x_t, x_{t+1}, \dots, x_{t+19}] \rightarrow x_{t+20}
$$

The input shape for PyTorch LSTM is usually:

$$
(\text{batch size}, \text{sequence length}, \text{number of features})
$$

Here we have one feature only: the sine value.

In [ ]:
def make_sequences(data, seq_len=20):
    X = []
    y = []
    for i in range(len(data) - seq_len):
        X.append(data[i:i + seq_len])
        y.append(data[i + seq_len])
    return np.array(X), np.array(y)

seq_len = 20
X, y = make_sequences(data, seq_len=seq_len)

# Add feature dimension: (samples, seq_len, features)
X = X[..., np.newaxis]
y = y[..., np.newaxis]

print("X shape:", X.shape)
print("y shape:", y.shape)

## 10. Train/test split

We will use the first 80% for training and the last 20% for testing.

For time series, this is better than random shuffling because future data should not leak into the past.

In [ ]:
train_size = int(0.8 * len(X))

X_train = X[:train_size]
y_train = y[:train_size]
X_test = X[train_size:]
y_test = y[train_size:]

# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

print("Training samples:", len(X_train_t))
print("Testing samples:", len(X_test_t))

## 11. Build a simple LSTM model

We use PyTorch's built-in `nn.LSTM`.

Important arguments:

- `input_size=1`: one feature per time step
- `hidden_size=16`: dimension of the hidden state
- `batch_first=True`: input shape is `(batch, sequence, features)`

The LSTM returns outputs for all time steps, but for next-step prediction we usually use the last output.

In [ ]:
class SimpleLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=16, output_size=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # lstm_out shape: (batch, seq_len, hidden_size)
        lstm_out, (h_n, c_n) = self.lstm(x)

        # Use the output from the final time step
        last_output = lstm_out[:, -1, :]

        # Map hidden state to prediction
        y_pred = self.fc(last_output)
        return y_pred

model = SimpleLSTM(input_size=1, hidden_size=16, output_size=1)
print(model)

## 12. Train the LSTM

We use mean squared error because this is a regression problem.

The loss is:

$$
\mathrm{MSE} = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2
$$

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

num_epochs = 60
train_losses = []

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)

    epoch_loss /= len(train_loader.dataset)
    train_losses.append(epoch_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1:3d}/{num_epochs}, Loss = {epoch_loss:.6f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses)
plt.title("Training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.show()

## 13. Evaluate the model

Now we compare the LSTM predictions with the true sine-wave values.

In [ ]:
model.eval()
with torch.no_grad():
    test_preds = model(X_test_t).numpy()

plt.figure(figsize=(10, 4))
plt.plot(y_test, label="True")
plt.plot(test_preds, label="Predicted")
plt.title("LSTM next-step prediction")
plt.xlabel("Test sample index")
plt.ylabel("Value")
plt.legend()
plt.show()

## 14. Inspect tensor shapes

Understanding shapes is very important before learning QLSTM.

For our input:

$$
X \in \mathbb{R}^{N \times T \times d}
$$

where:

- $N$ is the number of samples
- $T$ is the sequence length
- $d$ is the number of features

In this notebook:

- $T = 20$
- $d = 1$

In [ ]:
example_batch, example_target = next(iter(train_loader))

print("Example batch shape:", example_batch.shape)
print("Example target shape:", example_target.shape)

with torch.no_grad():
    lstm_out, (h_n, c_n) = model.lstm(example_batch)

print("LSTM output shape:", lstm_out.shape)
print("Final hidden state h_n shape:", h_n.shape)
print("Final cell state c_n shape:", c_n.shape)

## 15. How this prepares you for QLSTM

A classical LSTM gate uses a trainable transformation like:

$$
f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)
$$

A QLSTM changes the internal transformation.

Instead of using only a classical linear layer, a QLSTM may use a **variational quantum circuit** to process information.

A simplified view is:

$$
[h_{t-1}, x_t] \rightarrow \text{Quantum Circuit} \rightarrow \text{Expectation Values} \rightarrow \text{LSTM Gate}
$$

So before learning QLSTM, we should understand:

1. what $x_t$, $h_t$, and $c_t$ mean
2. what gates in an LSTM do
3. what trainable parameters mean
4. how a model maps input data to output values
5. how backpropagation updates parameters
6. how a quantum circuit can act as a trainable layer


## 16. Exercises

### Exercise 1

Change `seq_len` from 20 to 5.

Question: Does the model become better or worse? Why?

### Exercise 2

Change `hidden_size` from 16 to 4.

Question: What happens to the prediction quality?

### Exercise 3

Change `hidden_size` from 16 to 64.

Question: Does a bigger hidden size always help?

### Exercise 4

Try a different dataset:

$$
x_t = \sin(t) + 0.5\sin(3t)
$$

Question: Is this harder or easier than the simple sine wave?

### Exercise 5

Print the first input sequence and its target.

Question: What is the model trying to learn from this pair?


## 17. Exercise answers

### Answer 1

Using `seq_len = 5` usually gives the model less context.

For a smooth sine wave, it may still work, but it may predict less accurately because it sees a shorter history.

A longer sequence gives the model more information about the pattern.

### Answer 2

Using `hidden_size = 4` gives the LSTM less memory capacity.

The model may still learn the basic sine wave, but it may struggle more with complicated sequences.

### Answer 3

Using `hidden_size = 64` gives the model more capacity.

But bigger is not always better.

A larger model may train slower and may overfit if the dataset is small.

### Answer 4

The dataset

$$
x_t = \sin(t) + 0.5\sin(3t)
$$

is harder because it contains more than one frequency.

The model must learn both the slow oscillation and the faster oscillation.

### Answer 5

The first input sequence is a window of previous values.

The target is the next value immediately after that window.

So the model learns:

$$
[x_t, x_{t+1}, \dots, x_{t+T-1}] \rightarrow x_{t+T}
$$



## 18. Optional: manual view of LSTM gates

PyTorch's `nn.LSTM` hides the gate calculations inside the module.

Conceptually, the gates are:

$$
f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)
$$

$$
i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)
$$

$$
\tilde{c}_t = \tanh(W_c [h_{t-1}, x_t] + b_c)
$$

$$
o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)
$$

$$
c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t
$$

$$
h_t = o_t \odot \tanh(c_t)
$$

This is the most important conceptual bridge to QLSTM.

In QLSTM, the transformations that produce gate values can be partly replaced by quantum circuits.